# StandUp4AI Training - Word-Level Laughter Detection

This notebook trains XLM-RoBERTa-base for word-level laughter detection using the StandUp4AI dataset.

## Dataset
- **French**: 1052 words
- **English**: 1865 words (2 files)
- **Spanish**: 286 words
- **Total**: 3203 words with L (laughter) / O (other) labels

## 1. Setup

Install dependencies and mount Google Drive.

In [ ]:
# Install required packages
!pip install -q transformers torch scikit-learn pandas numpy

In [ ]:
# Mount Google Drive for saving checkpoints
from google.colab import drive
drive.mount('/content/drive')
import os

# Create output directory
OUTPUT_DIR = '/content/drive/MyDrive/standup4ai_models'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

## 2. Imports and Configuration

Define model config and import libraries.

In [ ]:
import torch
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
from transformers import AutoModel, AutoTokenizer, AdamW, get_linear_schedule_with_warmup
from torch.utils.data import Dataset, DataLoader
import warnings
warnings.filterwarnings('ignore')

# Configuration
MODEL_NAME = "xlm-roberta-base"
MAX_LEN = 64
BATCH_SIZE = 32
EPOCHS = 5
LR = 2e-5
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

## 3. Data Loading

Load the 4 CSV files from StandUp4AI dataset and assign language labels.

In [ ]:
# Data files with language mapping
# French: -1FrUOEswOk.csv (1052 words)
# English: 0g7nezWZyfY.csv (1054 words), 1xvwYZwm8Ig.csv (811 words)
# Spanish: 6JQzl2LlXbQ.csv (286 words)

from google.colab import files
import io

# Upload CSV files
print("Please upload the 4 StandUp4AI CSV files:")
uploaded = files.upload()

# Map filenames to language
file_lang_map = {
    '-1FrUOEswOk.csv': 'fr',
    '0g7nezWZyfY.csv': 'en',
    '1xvwYZwm8Ig.csv': 'en',
    '6JQzl2LlXbQ.csv': 'es'
}

dfs = []
for fname in uploaded.keys():
    if fname.endswith('.csv'):
        lang = file_lang_map.get(fname, 'unknown')
        df = pd.read_csv(io.BytesIO(uploaded[fname]))
        df['language'] = lang
        dfs.append(df)
        print(f"Loaded {fname}: {len(df)} rows, language={lang}")

data = pd.concat(dfs, ignore_index=True)
print(f"\nTotal samples: {len(data)}")

## 4. Data Preprocessing

Convert labels to binary (L=1 for laughter, O=0 for other).

In [ ]:
# Convert labels: L -> 1 (laughter), O -> 0 (other)
data['label'] = data['label'].map({'L': 1, 'O': 0})

# Check for any NaN labels
print(f"NaN in labels: {data['label'].isna().sum()}")

# Display class distribution
print("\nLabel distribution:")
print(data['label'].value_counts())

print("\nPer-language distribution:")
print(data.groupby('language')['label'].value_counts().unstack(fill_value=0))

## 5. Train/Val Split (Stratified by Language)

Split 80/20 for each language to maintain class balance.

In [ ]:
# Stratified split by language
train_dfs = []
val_dfs = []

for lang in data['language'].unique():
    lang_data = data[data['language'] == lang]
    train_lang, val_lang = train_test_split(
        lang_data,
        test_size=0.2,
        random_state=42,
        stratify=lang_data['label']
    )
    train_dfs.append(train_lang)
    val_dfs.append(val_lang)
    print(f"{lang}: train={len(train_lang)}, val={len(val_lang)}")

train_data = pd.concat(train_dfs, ignore_index=True)
val_data = pd.concat(val_dfs, ignore_index=True)

print(f"\nTotal: train={len(train_data)}, val={len(val_data)}")

## 6. Dataset Class

PyTorch Dataset for XLM-RoBERTa input.

In [ ]:
class LaughterDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(label, dtype=torch.long)
        }

## 7. Model Definition

XLM-RoBERTa-base + classification head.

In [ ]:
class XLMRLaughterClassifier(torch.nn.Module):
    def __init__(self, model_name, num_classes=2):
        super().__init__()
        self.xlmr = AutoModel.from_pretrained(model_name)
        self.dropout = torch.nn.Dropout(0.3)
        self.classifier = torch.nn.Linear(self.xlmr.config.hidden_size, num_classes)
    
    def forward(self, input_ids, attention_mask):
        outputs = self.xlmr(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.last_hidden_state[:, 0, :]  # CLS token
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
        return logits

## 8. Training Functions

In [ ]:
def train_epoch(model, data_loader, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    predictions = []
    actual_labels = []
    
    criterion = torch.nn.CrossEntropyLoss()
    
    for batch in data_loader:
        optimizer.zero_grad()
        
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        predictions.extend(preds)
        actual_labels.extend(labels.cpu().numpy())
    
    f1 = f1_score(actual_labels, predictions, average='weighted')
    return total_loss / len(data_loader), f1

def eval_epoch(model, data_loader, device):
    model.eval()
    total_loss = 0
    predictions = []
    actual_labels = []
    
    criterion = torch.nn.CrossEntropyLoss()
    
    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            
            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)
            
            total_loss += loss.item()
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            predictions.extend(preds)
            actual_labels.extend(labels.cpu().numpy())
    
    f1 = f1_score(actual_labels, predictions, average='weighted')
    return total_loss / len(data_loader), f1, predictions, actual_labels

## 9. Initialize Model and DataLoaders

In [ ]:
# Load tokenizer
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Create datasets
train_dataset = LaughterDataset(
    texts=train_data['text'].values,
    labels=train_data['label'].values,
    tokenizer=tokenizer,
    max_len=MAX_LEN
)

val_dataset = LaughterDataset(
    texts=val_data['text'].values,
    labels=val_data['label'].values,
    tokenizer=tokenizer,
    max_len=MAX_LEN
)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

# Initialize model
print(f"\nLoading model: {MODEL_NAME}")
model = XLMRLaughterClassifier(MODEL_NAME)
model = model.to(DEVICE)

# Optimizer and scheduler
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

print("Model initialized successfully!")

## 10. Training Loop

Train for 5 epochs with F1 monitoring.

In [ ]:
best_val_f1 = 0
history = {'train_loss': [], 'train_f1': [], 'val_loss': [], 'val_f1': []}

print("Starting training...\n")
for epoch in range(EPOCHS):
    print(f"Epoch {epoch + 1}/{EPOCHS}")
    print("-" * 40)
    
    # Train
    train_loss, train_f1 = train_epoch(model, train_loader, optimizer, scheduler, DEVICE)
    
    # Evaluate
    val_loss, val_f1, _, _ = eval_epoch(model, val_loader, DEVICE)
    
    history['train_loss'].append(train_loss)
    history['train_f1'].append(train_f1)
    history['val_loss'].append(val_loss)
    history['val_f1'].append(val_f1)
    
    print(f"Train Loss: {train_loss:.4f}, Train F1: {train_f1:.4f}")
    print(f"Val Loss: {val_loss:.4f}, Val F1: {val_f1:.4f}")
    
    # Save best model
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), f"{OUTPUT_DIR}/best_model.pt")
        print(f"  -> New best model saved! (F1: {best_val_f1:.4f})")
    
    print()

print(f"Training complete! Best Val F1: {best_val_f1:.4f}")

## 11. Per-Language Evaluation

Compute F1 scores for French, English, and Spanish separately.

In [ ]:
# Load best model for evaluation
model.load_state_dict(torch.load(f"{OUTPUT_DIR}/best_model.pt"))
model.eval()

# Get all predictions on validation set
_, _, predictions, actual_labels = eval_epoch(model, val_loader, DEVICE)

# Add predictions to val_data
val_data_eval = val_data.copy()
val_data_eval['predicted'] = predictions

# Per-language F1 scores
print("=" * 50)
print("PER-LANGUAGE EVALUATION RESULTS")
print("=" * 50)

results = {}
for lang in ['fr', 'en', 'es']:
    lang_data = val_data_eval[val_data_eval['language'] == lang]
    if len(lang_data) > 0:
        lang_f1 = f1_score(lang_data['label'], lang_data['predicted'], average='weighted')
        results[lang] = lang_f1
        print(f"\n{lang.upper()} (n={len(lang_data)}):")
        print(f"  F1 Score: {lang_f1:.4f}")
        
        # Per-class metrics
        print(f"  Classification Report:")
        report = classification_report(lang_data['label'], lang_data['predicted'], 
                                      target_names=['Other', 'Laughter'], digits=4)
        for line in report.split('\n'):
            print(f"    {line}")

print("\n" + "=" * 50)
print("SUMMARY")
print("=" * 50)
for lang, f1 in results.items():
    status = "PASS" if f1 >= 0.70 else "FAIL"
    print(f"{lang.upper()} F1 >= 0.70: {f1:.4f} [{status}]")

overall_f1 = f1_score(actual_labels, predictions, average='weighted')
status = "PASS" if overall_f1 >= 0.70 else "FAIL"
print(f"\nOVERALL Val F1 >= 0.70: {overall_f1:.4f} [{status}]")

## 12. Training Visualization

Plot loss and F1 curves.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss plot
axes[0].plot(history['train_loss'], label='Train Loss')
axes[0].plot(history['val_loss'], label='Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True)

# F1 plot
axes[1].plot(history['train_f1'], label='Train F1')
axes[1].plot(history['val_f1'], label='Val F1')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('F1 Score')
axes[1].set_title('Training and Validation F1')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/training_curves.png", dpi=150)
plt.show()
print(f"\nPlots saved to {OUTPUT_DIR}/training_curves.png")

## 13. Save Final Model and Tokenizer

In [ ]:
# Save the best model and tokenizer
model_path = f"{OUTPUT_DIR}/xlmr_laughter_classifier"

model.load_state_dict(torch.load(f"{OUTPUT_DIR}/best_model.pt"))
model.save_pretrained(model_path)
tokenizer.save_pretrained(model_path)

print(f"Model saved to: {model_path}")

# Save training history
import json
history_path = f"{OUTPUT_DIR}/training_history.json"
with open(history_path, 'w') as f:
    json.dump(history, f)
print(f"Training history saved to: {history_path}")

# Save per-language results
results_path = f"{OUTPUT_DIR}/per_language_results.json"
with open(results_path, 'w') as f:
    json.dump(results, f)
print(f"Per-language results saved to: {results_path}")

print("\n" + "=" * 50)
print("SAVED FILES")
print("=" * 50)
for f in os.listdir(OUTPUT_DIR):
    fpath = os.path.join(OUTPUT_DIR, f)
    size = os.path.getsize(fpath) / (1024 * 1024)  # MB
    print(f"  {f}: {size:.2f} MB")

## 14. Load and Use Model for Inference

Example inference on new text.

In [ ]:
# Load saved model
from transformers import AutoModelForSequenceClassification

inference_model = AutoModelForSequenceClassification.from_pretrained(model_path)
inference_model.to(DEVICE)
inference_model.eval()

def predict_laughter(text, threshold=0.5):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, 
                       max_length=MAX_LEN, padding=True)
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = inference_model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1)
        pred = torch.argmax(probs, dim=1).item()
        confidence = probs[0][pred].item()
    
    label = "Laughter" if pred == 1 else "Other"
    return label, confidence

# Test examples
test_texts = [
    "haha that was funny",
    "I think we should consider",
    "jaja que gracioso",
    "bonjour comment allez-vous"
]

print("INFERENCE EXAMPLES")
print("-" * 50)
for text in test_texts:
    label, conf = predict_laughter(text)
    print(f"Text: '{text}'")
    print(f"  -> {label} (confidence: {conf:.4f})\n")

---

## Summary

This notebook trained XLM-RoBERTa-base for word-level laughter detection on StandUp4AI data.

**Target Metrics:**
- French F1 >= 0.70
- English F1 >= 0.70
- Spanish F1 >= 0.70
- Overall Val F1 >= 0.70

**Output Files:**
- `xlmr_laughter_classifier/` - Saved model and tokenizer
- `best_model.pt` - Best checkpoint
- `training_curves.png` - Loss and F1 plots
- `training_history.json` - Epoch-by-epoch metrics
- `per_language_results.json` - Per-language F1 scores